# DeepAR Demand Forecasting

## Objective

The objective of this notebook is to train DeepAR as the project's
deep learning forecasting model.

DeepAR is trained as a global probabilistic forecasting model across
Store × Family time series.

An internal validation period is used for model selection and early stopping.
The final 15-day validation period (August 1–15, 2017) remains held out
for final evaluation.

## 1. Environment Setup

This notebook is designed to run in Google Colab for DeepAR training.

When running in Google Colab, Google Drive is mounted so the notebook can
access the project files and save trained model checkpoints.

When running locally, the Google Drive mounting cell should be skipped.
The project root is detected automatically from the repository structure.

Training was performed in Google Colab using a T4 GPU.

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Imports and Project Setup

In [3]:
import sys
import os
import numpy as np
import pandas as pd
import torch

In [10]:
from pathlib import Path
import sys
import os

# Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/DeepAR")
else:
    # Notebook is located inside Retail_Demand_Forec/notebooks/
    PROJECT_ROOT = Path.cwd().parent

SRC_PATH = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("=" * 60)
print("PROJECT SETUP")
print("=" * 60)
print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)
print("src exists:", SRC_PATH.exists())
print()

PROJECT SETUP
Environment: Local
PROJECT_ROOT: c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec
SRC_PATH: c:\Users\ino0i\OneDrive\سطح المكتب\Retail_Demand_Forec\src
src exists: True



In [11]:
from evaluation import (
    FORECAST_HORIZON,
    filter_evaluation_data,
    evaluate
)

print("✅ evaluation imported successfully")
print("FORECAST_HORIZON:", FORECAST_HORIZON)

✅ evaluation imported successfully
FORECAST_HORIZON: 15


In [12]:
#pip install
!pip install pytorch-forecasting lightning pyarrow


## 3. Load Data

We load the preprocessed train and validation datasets from the shared pipeline.

- `train_df`: 2013-01-30 → 2017-07-31 (2,364,490 rows)
- `valid_df`: 2017-08-01 → 2017-08-15 (25,929 rows)

In [13]:
train_df = pd.read_parquet(
    f"{PROJECT_ROOT}/data/processed/train_df.parquet"
)

valid_df = pd.read_parquet(
    f"{PROJECT_ROOT}/data/processed/valid_df.parquet"
)

## 4. Filter Final Evaluation Data

The shared evaluation framework excludes the Store 6 × BABY CARE series
from final evaluation because its prepared feature rows do not provide
the complete 15-day validation period.

This keeps the evaluation set consistent across all forecasting models.

In [14]:
valid_df_eval = filter_evaluation_data(valid_df)

print("valid_df shape:", valid_df.shape)
print("valid_df_eval shape:", valid_df_eval.shape)
print("Rows removed:", len(valid_df) - len(valid_df_eval))
print()
print("Date range:", valid_df_eval["date"].min(), "→", valid_df_eval["date"].max())
print("Unique series:", valid_df_eval.groupby(["store_nbr", "family"]).ngroups)
print("Unique stores:", valid_df_eval["store_nbr"].nunique())
print("Unique families:", valid_df_eval["family"].nunique())
print()
print("Sample:")
print(valid_df_eval[["date", "store_nbr", "family", "sales"]].head())

valid_df shape: (25929, 23)
valid_df_eval shape: (25920, 23)
Rows removed: 9

Date range: 2017-08-01 00:00:00 → 2017-08-15 00:00:00
Unique series: 1728
Unique stores: 54
Unique families: 33

Sample:
        date  store_nbr      family  sales
0 2017-08-01          1  AUTOMOTIVE    5.0
1 2017-08-02          1  AUTOMOTIVE    4.0
2 2017-08-03          1  AUTOMOTIVE    3.0
3 2017-08-04          1  AUTOMOTIVE    8.0
4 2017-08-05          1  AUTOMOTIVE    5.0


## 5. Check series length in train_df and valid_df_eval


In [15]:
train_series = (
    train_df.groupby(["store_nbr", "family"])["date"]
    .agg(["count", "min", "max"])
    .reset_index()
    .rename(columns={
        "count": "train_rows",
        "min": "train_min_date",
        "max": "train_max_date"
    })
)

valid_series = (
    valid_df_eval.groupby(["store_nbr", "family"])["date"]
    .agg(["count", "min", "max"])
    .reset_index()
    .rename(columns={
        "count": "valid_rows",
        "min": "valid_min_date",
        "max": "valid_max_date"
    })
)

print("=" * 60)
print("TRAIN DF")
print("=" * 60)
print("Number of series:", len(train_series))
print("Rows per series: min =", train_series["train_rows"].min(),
      "| max =", train_series["train_rows"].max(),
      "| mean =", int(train_series["train_rows"].mean()))
print()
print("=" * 60)
print("VALID DF EVAL")
print("=" * 60)
print("Number of series:", len(valid_series))
print("Rows per series: min =", valid_series["valid_rows"].min(),
      "| max =", valid_series["valid_rows"].max(),
      "| mean =", int(valid_series["valid_rows"].mean()))
print()
print("Series with less than 15 rows in valid:")
short_valid = valid_series[valid_series["valid_rows"] < 15]
print(short_valid.to_string(index=False))
print()
print("Series in valid but NOT in train:")
train_keys = set(zip(train_series["store_nbr"], train_series["family"]))
valid_keys = set(zip(valid_series["store_nbr"], valid_series["family"]))
missing_in_train = valid_keys - train_keys
print("Count:", len(missing_in_train))
print("These series:", sorted(missing_in_train)[:20])

TRAIN DF
Number of series: 1728
Rows per series: min = 75 | max = 1636 | mean = 1368

VALID DF EVAL
Number of series: 1728
Rows per series: min = 15 | max = 15 | mean = 15

Series with less than 15 rows in valid:
Empty DataFrame
Columns: [store_nbr, family, valid_rows, valid_min_date, valid_max_date]
Index: []

Series in valid but NOT in train:
Count: 0
These series: []


## 6. Combine train and valid for time_idx continuity
We need a continuous timeline per series so DeepAR sees the full history.



In [16]:
keep_cols = ["date", "store_nbr", "family", "sales", "onpromotion", "weekday", "month"]

train_small = train_df[keep_cols].copy()
valid_small = valid_df_eval[keep_cols].copy()

train_small["is_train"] = True
valid_small["is_train"] = False

df_all = pd.concat([train_small, valid_small], ignore_index=True)
df_all = df_all.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# Create series_id
df_all["series_id"] = (
    df_all["store_nbr"].astype(str)
    + "_"
    + df_all["family"]
)

# ✅ GLOBAL time_idx based on date
df_all["time_idx"] = (
    df_all["date"] - df_all["date"].min()
).dt.days

print("=" * 60)
print("COMBINED DF (with GLOBAL time_idx)")
print("=" * 60)
print("Shape:", df_all.shape)
print()
print("Time_idx range:", df_all["time_idx"].min(), "→", df_all["time_idx"].max())
print()
print("is_train counts:")
print(df_all["is_train"].value_counts())
print()
print("Per-series time_idx check:")
check = df_all.groupby("series_id")["time_idx"].agg(["min", "max", "count"])
print("Min time_idx across series:", check["min"].unique())
print("Max time_idx values:", check["max"].nunique(), "unique values")
print()
print("Sample:")
print(df_all.head(10).to_string())

COMBINED DF (with GLOBAL time_idx)
Shape: (2390410, 10)

Time_idx range: 0 → 1658

is_train counts:
is_train
True     2364490
False      25920
Name: count, dtype: int64

Per-series time_idx check:
Min time_idx across series: [   0  545  365   11  665  389  369  552  441   55    3  374  371 1134
    1   96  492  386  366  550   57  493  370 1222    6   66  388  549
  387  375  548  645  379  936  979  367  491  390  547  423    4    2
  932   53  666  607 1380   73  772  896  880  852  828  881  883  933
 1070  976 1010 1113 1569 1028 1054 1379 1223   51 1295   27  121  568
 1394  171  495 1290 1397  135 1318 1381  885 1405   62  394  807  882
  815  833 1291   63  556 1080  384 1196 1082   10  609   60 1068  127
 1086  129  424  131 1440    5  546   56  425  407 1337 1387   61  961
 1320  963 1002  573 1030 1375   54 1435 1378  512 1353  429 1340  720
 1392 1287]
Max time_idx values: 1 unique values

Sample:
        date  store_nbr      family  sales  onpromotion    weekday  month  is_

## 7. Data Distribution Checks

Before constructing the DeepAR datasets, we inspect the distribution of
the target (`sales`) and the known future covariate (`onpromotion`)
across the training and final validation periods.

This confirms the presence of zero-sales observations and the scale/skew
of the target used by the forecasting model.

In [17]:
print("=" * 60)
print("onpromotion distribution")
print("=" * 60)
print()
print("TRAIN period:")
print(df_all[df_all["is_train"] == True]["onpromotion"].describe())
print()
print("VALID period:")
print(df_all[df_all["is_train"] == False]["onpromotion"].describe())
print()
print("=" * 60)
print("Sales distribution (target)")
print("=" * 60)
print()
print("TRAIN:")
print(df_all[df_all["is_train"] == True]["sales"].describe())
print()
print("VALID:")
print(df_all[df_all["is_train"] == False]["sales"].describe())
print()
print("=" * 60)
print("Zero sales percentage")
print("=" * 60)
print()
print("TRAIN:", round((df_all[df_all['is_train'] == True]['sales'] == 0).mean() * 100, 2), "%")
print("VALID:", round((df_all[df_all['is_train'] == False]['sales'] == 0).mean() * 100, 2), "%")
print()
print("=" * 60)
print("Date ranges")
print("=" * 60)
print()
print("TRAIN:", df_all[df_all["is_train"] == True]["date"].min(),
      "→", df_all[df_all["is_train"] == True]["date"].max())
print("VALID:", df_all[df_all["is_train"] == False]["date"].min(),
      "→", df_all[df_all["is_train"] == False]["date"].max())

onpromotion distribution

TRAIN period:
count    2.364490e+06
mean     3.228184e+00
std      1.356019e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      7.410000e+02
Name: onpromotion, dtype: float64

VALID period:
count    25920.000000
mean         6.183565
std         16.354337
min          0.000000
25%          0.000000
50%          0.000000
75%          6.000000
max        244.000000
Name: onpromotion, dtype: float64

Sales distribution (target)

TRAIN:
count    2.364490e+06
mean     4.430356e+02
std      1.213826e+03
min      0.000000e+00
25%      3.000000e+00
50%      3.300000e+01
75%      2.905347e+02
max      1.247170e+05
Name: sales, dtype: float64

VALID:
count    25920.000000
mean       479.680669
std       1260.838463
min          0.000000
25%          5.000000
50%         34.000000
75%        291.790995
max      15190.000000
Name: sales, dtype: float64

Zero sales percentage

TRAIN: 15.55 %
VALID: 11.93 %

Date ranges

TRAI

## 8. Data Preparation for DeepAR

In [18]:
import warnings
warnings.filterwarnings("ignore")

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# ============================================================
# STEP 1: Convert categorical columns to string
# ============================================================
df_all["store_nbr"] = df_all["store_nbr"].astype(str)
df_all["family"] = df_all["family"].astype(str)
df_all["weekday"] = df_all["weekday"].astype(str)
df_all["month"] = df_all["month"].astype(str)

# ============================================================
# STEP 2: Parameters
# ============================================================
MAX_ENCODER_LENGTH = 30
MAX_PREDICTION_LENGTH = 15
INTERNAL_VALID_DAYS = 46

# ============================================================
# STEP 3: Split — 3 sets
# ============================================================
train_data_full = df_all[df_all["is_train"] == True].copy()
final_eval = df_all[df_all["is_train"] == False].copy()

train_max_date = train_data_full["date"].max()
internal_valid_start = train_max_date - pd.Timedelta(days=INTERNAL_VALID_DAYS - 1)

internal_train = train_data_full[
    train_data_full["date"] < internal_valid_start
].copy()

internal_valid = train_data_full[
    train_data_full["date"] >= internal_valid_start
].copy()

print("=" * 60)
print("INTERNAL SPLIT")
print("=" * 60)
print("Internal train shape:", internal_train.shape)
print("Internal valid shape:", internal_valid.shape)
print("Final eval shape:", final_eval.shape)
print()
print("Internal train dates:", internal_train["date"].min(), "→", internal_train["date"].max())
print("Internal valid dates:", internal_valid["date"].min(), "→", internal_valid["date"].max())
print("Final eval dates:", final_eval["date"].min(), "→", final_eval["date"].max())
print()
print("Internal train series:", internal_train["series_id"].nunique())
print("Internal valid series:", internal_valid["series_id"].nunique())
print("Final eval series:", final_eval["series_id"].nunique())
print()

# ============================================================
# STEP 4: Create training TimeSeriesDataSet
# ============================================================
training = TimeSeriesDataSet(
    internal_train,
    time_idx="time_idx",
    target="sales",
    group_ids=["series_id"],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["store_nbr", "family"],
    time_varying_known_categoricals=["weekday", "month"],
    time_varying_known_reals=["onpromotion"],
    time_varying_unknown_reals=["sales"],
    target_normalizer=GroupNormalizer(
        groups=["series_id"],
        transformation="log1p"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

# ============================================================
# STEP 5A: Internal validation (for Early Stopping)
# ============================================================
internal_train_max_time_idx = internal_train["time_idx"].max()

encoder_history_internal = internal_train[
    internal_train["time_idx"] > (internal_train_max_time_idx - MAX_ENCODER_LENGTH)
].copy()

internal_valid_with_context = pd.concat(
    [encoder_history_internal, internal_valid],
    ignore_index=True
).sort_values(["series_id", "time_idx"]).reset_index(drop=True)

validation_internal = TimeSeriesDataSet.from_dataset(
    training,
    internal_valid_with_context,
    predict=True,
    stop_randomization=True,
)

# ============================================================
# STEP 5B: Final evaluation (Aug 1-15) — UNTOUCHED
# ============================================================
# ⚠️ IMPORTANT: For final eval, we use train_data_full (ends 2017-07-31)
# for encoder history, because internal_train ends 2017-06-15,
# which leaves a gap before final_eval (2017-08-01).
train_data_full_max_time_idx = train_data_full["time_idx"].max()

encoder_history_final = train_data_full[
    train_data_full["time_idx"] > (train_data_full_max_time_idx - MAX_ENCODER_LENGTH)
].copy()

final_eval_with_context = pd.concat(
    [encoder_history_final, final_eval],
    ignore_index=True
).sort_values(["series_id", "time_idx"]).reset_index(drop=True)

validation_final = TimeSeriesDataSet.from_dataset(
    training,
    final_eval_with_context,
    predict=True,
    stop_randomization=True,
)

# ============================================================
# STEP 6: Verify
# ============================================================
print("=" * 60)
print("DATASETS CREATED")
print("=" * 60)
print("Training samples:", len(training))
print("Internal validation samples:", len(validation_internal))
print("Final evaluation samples:", len(validation_final))
print()
print("Encoder history (internal):", encoder_history_internal.shape)
print("Internal valid + context:", internal_valid_with_context.shape)
print()
print("Encoder history (final):", encoder_history_final.shape)
print("Final eval + context:", final_eval_with_context.shape)

INTERNAL SPLIT
Internal train shape: (2285002, 10)
Internal valid shape: (79488, 10)
Final eval shape: (25920, 10)

Internal train dates: 2013-01-30 00:00:00 → 2017-06-15 00:00:00
Internal valid dates: 2017-06-16 00:00:00 → 2017-07-31 00:00:00
Final eval dates: 2017-08-01 00:00:00 → 2017-08-15 00:00:00

Internal train series: 1728
Internal valid series: 1728
Final eval series: 1728

DATASETS CREATED
Training samples: 2320281
Internal validation samples: 1728
Final evaluation samples: 1728

Encoder history (internal): (51808, 10)
Internal valid + context: (131296, 10)

Encoder history (final): (51840, 10)
Final eval + context: (77760, 10)


## 9. BUILD DeepAR MODEL

In [19]:
from pytorch_forecasting.models import DeepAR
from pytorch_forecasting.metrics import NormalDistributionLoss

deepar = DeepAR.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=NormalDistributionLoss(),
)

print("DeepAR model created")
print()
total_params = sum(p.numel() for p in deepar.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

DeepAR model created

Total trainable parameters: 19,650


## 10. BUILD DATALOADERS


In [20]:
print("=" * 60)
print("REBUILDING DATALOADERS")
print("=" * 60)
print()

# 1. Train dataloader (on internal_train)
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=512,
    num_workers=0
)

# 2. Internal valid dataloader (for Early Stopping)
val_dataloader_internal = validation_internal.to_dataloader(
    train=False,
    batch_size=512,
    num_workers=0
)

# 3. Final eval dataloader (for final prediction only)
val_dataloader_final = validation_final.to_dataloader(
    train=False,
    batch_size=512,
    num_workers=0
)
print("Train dataloader batches:", len(train_dataloader))
print("Internal valid dataloader batches:", len(val_dataloader_internal))
print("Final eval dataloader batches:", len(val_dataloader_final))
print()
print("=" * 60)
print("DATALOADERS READY")
print("=" * 60)

REBUILDING DATALOADERS

Train dataloader batches: 4531
Internal valid dataloader batches: 4
Final eval dataloader batches: 4

DATALOADERS READY


## 11. Configure Trainer

In [21]:
# ============================================================
# REBUILD TRAINER (with correct dataloaders)
# ============================================================
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor

# Callbacks
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=5,
    verbose=True,
    mode="min"
)

lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, lr_logger],
    enable_model_summary=True,
    log_every_n_steps=100,
)

print("Trainer configured")
print("Max epochs:", trainer.max_epochs)
print("Accelerator:", trainer.accelerator)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Trainer configured
Max epochs: 5
Accelerator: <lightning.pytorch.accelerators.cpu.CPUAccelerator object at 0x000001E60D8F2C30>


> **Training note**
>
> The following cells perform actual DeepAR training and may require a
> GPU-enabled environment such as Google Colab with a T4 GPU.
>
> Phase 1 is used for internal validation and training-duration selection.
> Phase 2 retrains the final model from scratch on the complete training
> period.
>
> These cells do not need to be rerun when using the saved final checkpoint
> for inference and evaluation.

## PHASE 1: TRAIN DeePAR



In [ ]:
print("=" * 60)
print("STARTING DEEPAR TRAINING")
print("=" * 60)
print()
print("Train batches per epoch:", len(train_dataloader))
print("Internal valid batches:", len(val_dataloader_internal))
print("Max epochs:", trainer.max_epochs)
print()
print("Note: EarlyStopping will stop if val_loss doesn't improve for 5 epochs.")
print()

trainer.fit(
    deepar,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader_internal,  # ← Internal (NOT final)
)

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print("Best model path:", trainer.checkpoint_callback.best_model_path)
print("Best val_loss:", trainer.checkpoint_callback.best_model_score)

In [ ]:
# ============================================================
# SAVE BEST PHASE 1 CHECKPOINT
# ============================================================
import shutil
import os

best_model_path = trainer.checkpoint_callback.best_model_path

checkpoint_dir = PROJECT_ROOT / "models"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_filename = os.path.basename(best_model_path)
checkpoint_drive_path = checkpoint_dir / checkpoint_filename

shutil.copy2(best_model_path, checkpoint_drive_path)

print("Best Phase 1 checkpoint:", checkpoint_filename)
print("Saved to:", checkpoint_drive_path)

if checkpoint_drive_path.exists():
    size_mb = checkpoint_drive_path.stat().st_size / (1024 * 1024)
    print(f"✅ Checkpoint saved successfully ({size_mb:.2f} MB)")

### Phase 1 Result

Phase 1 established the DeepAR configuration and training duration using
chronological internal validation.

The best checkpoint was selected based on internal validation loss.
The final validation period (August 1–15, 2017) remained completely
unseen during this phase.

The selected training duration was then used for final retraining on
the complete training period.

## Phase 2 — Retrain on Full Training Data

After Phase 1, the training duration was selected based on internal
validation performance.

The final DeepAR model is retrained from scratch using the complete
training period:

**2013-01-30 → 2017-07-31**

The final validation period:

**2017-08-01 → 2017-08-15**

remains completely unseen during training.

The final model uses the same architecture and hyperparameters as Phase 1
and is trained for 4 epochs based on the Phase 1 result. Retrain DeepAR from scratch using the full training period
 (2013-01-30 → 2017-07-31). The number of epochs (4) was selected from Phase 1 based on internal validation performance. The final validation period (2017-08-01 → 2017-08-15) remains completely unseen during training.

In [ ]:
print("=" * 60)
print("RETRAINING ON FULL DATA")
print("=" * 60)
print()

# 1. Create TimeSeriesDataSet on the full training period
training_full = TimeSeriesDataSet(
    train_data_full,
    time_idx="time_idx",
    target="sales",
    group_ids=["series_id"],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["store_nbr", "family"],
    time_varying_known_categoricals=["weekday", "month"],
    time_varying_known_reals=["onpromotion"],
    time_varying_unknown_reals=["sales"],
    target_normalizer=GroupNormalizer(
        groups=["series_id"],
        transformation="log1p"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

print("Full training samples:", len(training_full))

# 2. Create full-data dataloader
train_dataloader_full = training_full.to_dataloader(
    train=True,
    batch_size=512,
    num_workers=0,
)

print("Full train batches per epoch:", len(train_dataloader_full))
print()

# 3. Build DeepAR from scratch
deepar_full = DeepAR.from_dataset(
    training_full,
    learning_rate=1e-3,
    hidden_size=32,
    rnn_layers=2,
    dropout=0.1,
    loss=NormalDistributionLoss(),
)

print("DeepAR model (full data) created")

# 4. Train using the selected epoch count
trainer_full = pl.Trainer(
    max_epochs=4,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    enable_model_summary=True,
    log_every_n_steps=100,
)

print()
print("Starting retraining on full data...")
print()

trainer_full.fit(
    deepar_full,
    train_dataloaders=train_dataloader_full,
)

print()
print("=" * 60)
print("RETRAINING COMPLETE")
print("=" * 60)
print("Best model path:", trainer_full.checkpoint_callback.best_model_path)

RETRAINING ON FULL DATA

Full training samples: 2400281
Full train batches per epoch: 4688



INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

DeepAR model (full data) created

Starting retraining on full data...



┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                   ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                   │ NormalDistributionLoss │      0 │ train │     0 │
│ 1 │ logging_metrics        │ ModuleList             │      0 │ train │     0 │
│ 2 │ embeddings             │ MultiEmbedding         │  1.3 K │ train │     0 │
│ 3 │ rnn                    │ LSTM                   │ 18.3 K │ train │     0 │
│ 4 │ distribution_projector │ Linear                 │     66 │ train │     0 │
└───┴────────────────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 19.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 19.7 K                                                                                               
Total estimated model params size (MB): 0.079                                                                      
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=4` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=4` reached.



RETRAINING COMPLETE
Best model path: /content/lightning_logs/version_2/checkpoints/epoch=3-step=18752.ckpt


### Phase 2 Result

After selecting the training duration in Phase 1, DeepAR was retrained
from scratch using the complete training period.

The final validation period was kept completely unseen during training.

The model architecture and hyperparameters were unchanged from Phase 1,
while the model was retrained using all available training observations.

## SAVE FINAL DEEPAR MODEL

In [ ]:
# ============================================================
# SAVE FINAL DEEPAR MODEL
# ============================================================
import shutil

best_full_path = trainer_full.checkpoint_callback.best_model_path

full_model_path = PROJECT_ROOT / "models" / "deepar_full_trained.ckpt"

shutil.copy2(
    best_full_path,
    full_model_path
)

print("Final checkpoint:", best_full_path)
print("Saved to:", full_model_path)

if full_model_path.exists():
    size_mb = full_model_path.stat().st_size / (1024 * 1024)
    print(f"✅ Final DeepAR model saved ({size_mb:.2f} MB)")

### Final DeepAR Model

The final DeepAR model was retrained on the complete training period
(2013-01-30 → 2017-07-31) using the training duration selected in Phase 1.

The final checkpoint was saved as:

`models/deepar_full_trained.ckpt`

This checkpoint is used by the inference and final evaluation workflow.

### Project Structure

The project uses the following main components:

- `data/processed/` — prepared training and validation datasets
- `models/` — trained DeepAR checkpoints
- `predictions/` — DeepAR forecast outputs
- `src/` — shared evaluation utilities
- `notebooks/` — modeling and evaluation workflows